# Feature Engineering Parque Solar Girasol

#### 1. IMPORTS Y CONFIGURACIÓN

In [1]:
import os
import pandas as pd
import numpy as np
import math

from statsmodels.tsa.stattools import adfuller, kpss
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

#### 2. FUNCIONES AUXILIARES

In [2]:
def identify_non_stationary(df, alpha=0.05):
    non_stat = []
    for col in df.select_dtypes('number').columns:
        s = df[col].dropna()
        if len(s) < 10:
            continue
        try:
            p_adf = adfuller(s)[1]
            p_kpss = kpss(s, nlags='auto')[1]
        except:
            non_stat.append(col)
            continue
        if p_adf >= alpha or p_kpss <= alpha:
            non_stat.append(col)
    return non_stat

def compute_best_lags(df, target='generation', max_lag=24):
    vars_ = [c for c in df.select_dtypes('number').columns if c != target]
    xcorr = pd.DataFrame({
        v: [df[target].corr(df[v].shift(l)) for l in range(max_lag+1)]
        for v in vars_
    }, index=range(max_lag+1))
    return {v: int(xcorr[v].abs().idxmax()) for v in vars_}

def add_temporal_features(df):
    df = df.copy()
    idx = df.index
    df['hour']     = idx.hour
    df['hour_sin'] = np.sin(2*np.pi * df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi * df['hour']/24)
    df['dow']      = idx.dayofweek
    df['dow_sin']  = np.sin(2*np.pi * df['dow']/7)
    df['dow_cos']  = np.cos(2*np.pi * df['dow']/7)
    df['month']    = idx.month
    df['month_sin']= np.sin(2*np.pi * df['month']/12)
    df['month_cos']= np.cos(2*np.pi * df['month']/12)
    return df

#### 3. TRANSFORMER PERSONALIZADO

In [3]:
class SolarFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self,
                 target='generation',
                 max_lag=24,
                 roll_windows=None,
                 log_transform_cols=None):
        self.target = target
        self.max_lag = max_lag
        self.roll_windows = roll_windows or [3,6,24]
        self.log_transform_cols = log_transform_cols or []

    def fit(self, X, y=None):
        self.non_stat_vars_ = identify_non_stationary(X)
        self.best_lags_    = compute_best_lags(X, target=self.target, max_lag=self.max_lag)
        return self

    def transform(self, X):
        df = X.copy()

        # A) Log-transform del target
        #df[f"{self.target}_log1p"] = np.log1p(df[self.target])

        # B) Clip outliers
        clip_q = [0.001, 0.999]
        num = df.select_dtypes('number').columns.drop(
            [self.target, f"{self.target}_log1p"], errors='ignore'
        )
        lowers = df[num].quantile(clip_q[0])
        uppers = df[num].quantile(clip_q[1])
        df[num] = df[num].clip(lower=lowers, upper=uppers, axis=1)

        # 1) Diferencias
        for col in self.non_stat_vars_:
            df[f"{col}_diff1"] = df[col].diff(1)

        # 2) Lags óptimos y 24h
        for col, lag in self.best_lags_.items():
            if col == self.target: continue
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
        for col in num:
            df[f"{col}_lag24"] = df[col].shift(24)

        # 3) Rolling means
        numeric = df.select_dtypes('number').columns.drop(
            [self.target, f"{self.target}_log1p"], errors='ignore'
        )
        for w in self.roll_windows:
            for col in numeric:
                df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()

        # 4) Interacciones y ratios
        for col in ['shortwave_radiation', 'global_tilted_irradiance']:
            if col in df:
                df[f"{col}_sq"] = df[col]**2
        if {'diffuse_radiation','global_tilted_irradiance'}.issubset(df.columns):
            df['diffuse_ratio'] = df['diffuse_radiation'] / (df['global_tilted_irradiance']+1e-6)

        # 5) Temporales cíclicos
        df = add_temporal_features(df)

        # 6) Log-transform extras
        for col in self.log_transform_cols:
            if col in df:
                df[f"{col}_log1p"] = np.log1p(df[col])

        # 7) Borramos cruft: las columnas originales que ya no usamos
        drop_cols = list(self.non_stat_vars_)
        df = df.drop(columns=drop_cols, errors='ignore')

        # 8) Filtrar NaNs y resetear índice si quieres
        df = df.dropna()

        # 9) Guardar feature names para usar luego en producción
        self.feature_names_ = df.columns.tolist()
        return df

    def get_feature_names_out(self):
        return self.feature_names_


#### 4. CARGA DE DATOS LIMPIOS

In [4]:
input_path = "../data/interim/meteo_data_with_generation_clean/parque_solar_girasol_clean.parquet"
df = pd.read_parquet(input_path)

#### 5. CONFIGURACIÓN DE FEATURE ENGINEERING

In [5]:
log_cols = [
    'shortwave_radiation','diffuse_radiation',
    'global_tilted_irradiance','shortwave_radiation_instant',
    'diffuse_radiation_instant','global_tilted_irradiance_instant',
    'direct_radiation','direct_normal_irradiance',
    'wind_speed_10m','wind_gusts_10m',
    'vapour_pressure_deficit','sunshine_duration'
]

sfe = SolarFeatureEngineer(
    target='generation',
    max_lag=24,
    roll_windows=[3,6,24],
    log_transform_cols=log_cols
)

# Entrenas tu transformer con los datos históricos:
sfe.fit(df)

# Y luego generas ya solo el dataset de features listo para modelar:
df_feat = sfe.transform(df)

# Si necesitas ver el listado final de columnas:
sfe.get_feature_names_out()

C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  p_kpss = kpss(s, nlags='auto')[1]
C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  p_kpss = kpss(s, nlags='auto')[1]
C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  p_kpss = kpss(s, nlags='auto')[1]
C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\443034564.py:9: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value

['date',
 'generation',
 'cloud_cover',
 'cloud_cover_low',
 'cloud_cover_mid',
 'et0_fao_evapotranspiration',
 'sunshine_duration',
 'shortwave_radiation',
 'global_tilted_irradiance',
 'shortwave_radiation_instant',
 'global_tilted_irradiance_instant',
 'direct_radiation',
 'direct_normal_irradiance',
 'terrestrial_radiation',
 'direct_radiation_instant',
 'direct_normal_irradiance_instant',
 'terrestrial_radiation_instant',
 'temperature_2m_diff1',
 'dew_point_2m_diff1',
 'relative_humidity_2m_diff1',
 'apparent_temperature_diff1',
 'surface_pressure_diff1',
 'vapour_pressure_deficit_diff1',
 'wind_speed_10m_diff1',
 'wind_direction_10m_diff1',
 'wind_gusts_10m_diff1',
 'is_day_diff1',
 'wet_bulb_temperature_2m_diff1',
 'diffuse_radiation_diff1',
 'diffuse_radiation_instant_diff1',
 'pressure_msl_diff1',
 'temperature_2m_lag0',
 'dew_point_2m_lag4',
 'relative_humidity_2m_lag0',
 'apparent_temperature_lag0',
 'surface_pressure_lag3',
 'cloud_cover_lag6',
 'cloud_cover_low_lag15',
 '

#### 6. GENERACIÓN DEL DATASET DE FEATURES

In [6]:
df_feat = sfe.transform(df)

C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\2176273940.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()
C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\2176273940.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_roll{w}h"] = df[col].rolling(window=w, min_periods=1).mean()
C:\Users\ferna\AppData\Local\Temp\ipykernel_23796\2176273940.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fr

#### 7. GUARDADO DEL DATASET PROCESADO

In [7]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "parque_solar_girasol_model_ready.parquet")
df_feat.to_parquet(output_path)

print(f"Features generadas y guardadas en: {output_path}")

Features generadas y guardadas en: ../data/processed\parque_solar_girasol_model_ready.parquet
